# AlgoCourse Lecture 2: Statistical Arbitrage
**Imperial Algorithmic Trading Society**

Tonight we follow one trade, **EWA against EWC**: two ETFs holding the largest companies of Australia (EWA) and Canada (EWC). We hedge one with the other, test whether the gap between them comes back, then backtest a trading rule. There are four short tasks, each matching a **Notebook task** slide in the lecture.

**How to use this notebook**
- Run the setup cell first (click it, then Shift + Enter). It loads the data from the course GitHub.
- In each task, replace every `___` with the right code, then run the cell.
- Each task lists the functions you'll need. If you get stuck, we show the solution after each task.
- To keep your work, use **File → Save a copy in Drive**.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm
from statsmodels.tsa.stattools import adfuller, acf
import warnings; warnings.filterwarnings("ignore", category=FutureWarning)

URL = "https://raw.githubusercontent.com/ekaterinaveter/AlgoCourse-2026/main/ewa_ewc.csv"
prices = pd.read_csv(URL, index_col="Date", parse_dates=True).dropna()   # daily closing prices, US dollars
prices.tail()

## Task 1: returns and the hedge ratio
**Why:** to hedge EWA with EWC we need β: how much EWA moves, on average, when EWC moves 1%. Fit it on 2015 to 2019.

1. Compute daily log returns, $r_t = \ln(P_t / P_{t-1})$.
2. Regress EWA's returns on EWC's: $r_{EWA,t} = \alpha + \beta\, r_{EWC,t} + \varepsilon_t$. The slope is β.
3. Compare the volatility of EWA alone with the hedged return, $r_{EWA} - \beta\, r_{EWC}$.

**Functions you'll need**
| What | Code |
|---|---|
| Log of every price | `np.log(prices)` |
| Change from the day before | `.diff()` |
| Regression of y on x with a constant | `sm.OLS(y, sm.add_constant(x)).fit()` |
| Its slope | `model.params["EWC"]` |
| Volatility (annualised) | `series.std() * np.sqrt(252)` |

In [ ]:
# 1. Daily log returns
returns = np.log(prices).___().dropna()
r = returns.loc[:"2019"]                                  # the data we fit on

# 2. Regress EWA's returns on EWC's
model = sm.OLS(r["EWA"], sm.add_constant(r[___])).fit()
beta = model.params["EWC"]
print(f"beta: {beta:.2f}   (EWC moves 1%, EWA moves {beta:.2f}% on average)")

# 3. Volatility before and after the hedge
hedged = r["EWA"] - ___ * r["EWC"]
print(f"Volatility of EWA alone: {r['EWA'].std() * np.sqrt(252):.0%}")
print(f"Volatility hedged:         {hedged.___() * np.sqrt(___):.0%}")

# Scatter plot with the fitted line (already written for you)
plt.figure(figsize=(5, 5))
plt.scatter(100 * r["EWC"], 100 * r["EWA"], s=3, alpha=0.5)
xs = np.linspace(-8, 8, 2)
plt.plot(xs, 100 * model.params["const"] + beta * xs, color="red")
plt.xlabel("EWC daily return (%)"); plt.ylabel("EWA daily return (%)"); plt.show()

## Task 2: stationarity and autocorrelation
**Why:** statistics uses the past to say something about the future. That only works on a series that behaves the same way over time (stationary), and a signal needs the past to predict the future (autocorrelation).

1. Look at the plots of prices and daily returns. Which look stationary?
2. Build the spread $s_t = \ln \text{EWA}_t - \beta \ln \text{EWC}_t$, with β from Task 1. Adding up the hedged daily returns gives exactly this.
3. Using 2015 to 2019, compute $\rho(1)$ to $\rho(10)$ for EWA's returns, EWC's returns and the spread. Which one can you predict from its past?

**Functions you'll need**
| What | Code |
|---|---|
| Log prices | `logp = np.log(prices)` |
| $\rho(1)$ to $\rho(10)$ of a series | `acf(series, nlags=10)[1:]` |

In [ ]:
# 1. Prices and returns (already written for you)
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
(100 * prices / prices.iloc[0]).plot(ax=ax[0], title="Prices, rebased to 100 at the start")
(100 * returns).plot(ax=ax[1], title="Daily returns (%)", lw=0.5, alpha=0.6)
plt.show()

# 2. The spread, for the whole period
logp = np.log(prices)
spread = logp["EWA"] - ___ * logp["EWC"]
s = spread.loc[:"2019"]                                   # the part we fit on

# 3. Autocorrelation, 2015 to 2019
acf_ewa  = acf(___, nlags=10)[1:]
acf_ewc     = acf(___, nlags=10)[1:]
acf_spread = acf(___, nlags=10)[1:]

fig, ax = plt.subplots(1, 3, figsize=(12, 3), sharey=True)
for a, values, title in zip(ax, [acf_ewa, acf_ewc, acf_spread], ["EWA returns", "EWC returns", "Spread"]):
    a.bar(range(1, 11), values); a.axhline(0, color="grey"); a.set_title(title); a.set_xlabel("lag k")
plt.show()
print("rho(1): EWA", acf_ewa[0].round(2), " EWC", acf_ewc[0].round(2), " spread", acf_spread[0].round(2))

## Task 3: test the spread
**Why:** the bar chart suggests the spread comes back. A statistical test tells us whether we can conclude it is stationary, rather than just looking like it.

The test's null hypothesis H₀ is "the spread is not stationary". If the p-value is below 0.05, we reject H₀ and conclude the spread is stationary.

1. Test the spread on 2015 to 2019 and read the p-value.
2. Compute the z-score, $z_t = (s_t - \text{mean}) / \text{std}$, using the mean and std from 2015 to 2019. It says how many standard deviations the spread is from its average.
3. Plot the z-score with lines at +2 and −2.

**Functions you'll need**
| What | Code |
|---|---|
| Stationarity test: the p-value | `adfuller(series)[1]` |
| Average and standard deviation | `s.mean()`, `s.std()` |

In [ ]:
# 1. Stationarity test on 2015 to 2019
pvalue = adfuller(___)[1]
print(f"p-value: {pvalue:.3f}  (below 0.05 means the spread is stationary)")

# 2. z-score for the whole period, using the 2015 to 2019 mean and std
z = (spread - s.___()) / s.___()

# 3. Plot (already written for you)
z.loc[:"2019"].plot(figsize=(10, 4), title="z-score of the spread, 2015 to 2019")
for level in (-2, 2):
    plt.axhline(level, color="red", linestyle="--")
plt.axhline(0, color="grey"); plt.show()

## Task 4: backtest the rule, twice
**Why:** a backtest runs the rule on past data to see what it would have made. We run it on two periods:
- **2015 to 2019**, the same years we fitted β, the mean and the std on;
- **2020 onwards**, data the rule has never seen. Don't refit anything.

The rule: if $z > +2$, sell the spread (position $-1$: sell EWA, buy EWC). If $z < -2$, buy it ($+1$). Close when $z$ crosses back through 0.

1. Fill in the rule in `positions`.
2. Daily profit = **yesterday's** position × today's change in the spread. (Why yesterday's?)
3. Run both backtests and rerun the stationarity test on 2020 onwards. Which backtest should you believe?

**Functions you'll need**
| What | Code |
|---|---|
| Yesterday's position | `pos.shift(1)` |
| Today's change in the spread | `spread.diff()` |
| Sharpe ratio (annualised) | `ret.mean() / ret.std() * np.sqrt(252)` |
| Data from 2020 onwards | `.loc["2020":]` |

In [ ]:
def positions(z, entry=2.0, exit=0.0):
    pos, current = [], 0
    for zi in z:
        if current == 0 and zi > entry:
            current = ___          # sell the spread
        elif current == 0 and zi < -entry:
            current = ___          # buy the spread
        elif current == 1 and zi >= exit:
            current = 0
        elif current == -1 and zi <= exit:
            current = 0
        pos.append(current)
    return pd.Series(pos, index=z.index)


def backtest(z, spread, cost_bp=5.0):
    pos = positions(z)
    pnl = (pos.___(1) * spread.diff()).fillna(0)             # yesterday's position x today's change
    costs = pos.diff().abs().fillna(0) * (1 + beta) * cost_bp / 1e4
    ret = (pnl - costs) / (1 + beta)                           # return on the money in both legs
    cum = ret.cumsum()
    (100 * cum).plot(figsize=(10, 3), title="Cumulative return (%)"); plt.show()
    return {"Sharpe": round(float(ret.mean() / ret.std() * np.sqrt(252)), 2),
            "max drawdown": f"{(cum - cum.cummax()).min():.1%}",
            "trades": int(((pos != 0) & (pos.shift(1).fillna(0) == 0)).sum())}


# 2015 to 2019: the data we fitted on
print("2015 to 2019:", backtest(z.loc[:"2019"], spread.loc[:"2019"]))

# 2020 onwards: new data, nothing refitted
print("2020 onwards:", backtest(z.loc[___], spread.loc[___]))

# Stationarity test on 2020 onwards
print(f"p-value, 2020 onwards: {adfuller(spread.loc[___])[1]:.2f}")

# The z-score for the whole period (already written for you)
ax = z.loc[:"2019"].plot(figsize=(10, 4), label="fitted: 2015 to 2019")
z.loc["2020":].plot(ax=ax, color="red", label="new data: 2020 onwards")
for level in (-2, 2):
    ax.axhline(level, color="grey", linestyle="--")
ax.legend(); plt.show()
print(f"z today: {z.iloc[-1]:.1f}")